# 🚀 Career Operating System — Interactive CV Generator

Welcome to the **Interactive CV Generator**. 
Run **Cell 1** once to load project libraries, then use the unified UI in **Cell 2** to select parameters, paste your Job Description, and click **🚀 Generate Tailored CV**.

In [1]:
import os
import sys
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, Markdown

# Ensure project root and src/ are in Python path, and CWD is project root
project_root = Path.cwd()
if project_root.name == 'notebooks' or not (project_root / 'src').is_dir():
    if (project_root.parent / 'src').is_dir():
        project_root = project_root.parent
        os.chdir(project_root)

if str(project_root / 'src') not in sys.path:
    sys.path.insert(0, str(project_root / 'src'))

print(f'✅ Project environment and path initialized at: {project_root}')


✅ Project environment and path initialized at: c:\Users\bvrabete\source\personal\career-os


## ⚙️ Interactive Generator Interface

Configure your run options below, paste your target Job Description, and click **🚀 Generate Tailored CV**.

In [2]:
import logging
logger = logging.getLogger('notebook_ui')
logging.basicConfig(level=logging.INFO, format='[%(levelname)s] %(message)s')
for noisy_lib in ['httpx', 'httpcore', 'openai', 'urllib3', 'google', 'google_genai']:
    logging.getLogger(noisy_lib).setLevel(logging.WARNING)
from generation import build_graph
from pdf_generator import generate_pdf
from docx_generator import generate_docx
from utils import validate_path
from kb_config import get_wiki_dir, get_output_dir, set_wiki_dir

# 1. Configuration & Parameter Widgets
config_picker = widgets.Dropdown(
    options=[
        ("Default Config (config.yaml)", "config.yaml"),
        ("OpenAI Config (config.openai.yaml)", "config.openai.yaml"),
        ("Google Gemini Config (config.gemini.yaml)", "config.gemini.yaml"),
        ("Hybrid Gemini Config (config.hybrid-gemini.yaml)", "config.hybrid-gemini.yaml"),
        ("Local Heavy Config (config.local-heavy.yaml)", "config.local-heavy.yaml"),
    ],
    value="config.yaml",
    description="Config File:",
    style={'description_width': '140px'},
    layout=widgets.Layout(width='650px')
)

strategy_picker = widgets.Dropdown(
    options=[
        ("Auto-detect (Recommended)", None),
        ("EMEA (European / Corporate)", "emea"),
        ("US Tech (Silicon Valley / High Density)", "us_tech"),
        ("Executive (Leadership / Strategic)", "executive"),
    ],
    value=None,
    description="Strategy Override:",
    style={'description_width': '140px'},
    layout=widgets.Layout(width='650px')
)

wiki_dir_input = widgets.Text(
    value=str(get_wiki_dir()),
    description="LLM-Wiki Directory:",
    style={'description_width': '140px'},
    layout=widgets.Layout(width='650px')
)

pdf_checkbox = widgets.Checkbox(
    value=True,
    description="Generate PDF Output (.pdf)",
    style={'description_width': 'initial'}
)

docx_checkbox = widgets.Checkbox(
    value=False,
    description="Generate Word Output (.docx)",
    style={'description_width': 'initial'}
)

# 2. Target Job Description & Output Filename Inputs
jd_file_input = widgets.Text(
    value="",
    placeholder="Optional: enter path to JD file (e.g. job-descriptions/target_jd.txt)",
    description="JD File Path:",
    style={'description_width': '140px'},
    layout=widgets.Layout(width='500px')
)

load_jd_file_btn = widgets.Button(
    description="📂 Load File",
    button_style="info",
    layout=widgets.Layout(width='140px')
)

jd_input = widgets.Textarea(
    value="",
    placeholder="Paste the complete target Job Description text here (or load from file above)...",
    description="Job Description:",
    style={'description_width': '140px'},
    layout=widgets.Layout(width='650px', height='220px')
)

def on_load_jd_file(b):
    path_str = jd_file_input.value.strip()
    if not path_str:
        print("⚠️ Please enter a JD file path to load.")
        return
    p = Path(path_str)
    if not p.is_file():
        print(f"❌ File not found: {p}")
        return
    try:
        content = p.read_text(encoding="utf-8")
        jd_input.value = content
        print(f"✅ Loaded Job Description from {p} ({len(content)} chars)")
    except Exception as e:
        print(f"❌ Error reading file: {e}")

load_jd_file_btn.on_click(on_load_jd_file)


output_name_input = widgets.Text(
    value="Notebook_Output_CV.md",
    description="Output Filename:",
    style={'description_width': '140px'},
    layout=widgets.Layout(width='650px')
)

# 3. Button and Output Area
generate_button = widgets.Button(
    description="🚀 Generate Tailored CV",
    button_style="success",
    icon="play",
    layout=widgets.Layout(width='320px', height='48px')
)

out_area = widgets.Output()

def on_generate_clicked(b):
    with out_area:
        out_area.clear_output()
        
        jd_text = jd_input.value.strip()
        if not jd_text and jd_file_input.value.strip():
            p = Path(jd_file_input.value.strip())
            if p.is_file():
                jd_text = p.read_text(encoding="utf-8")
                jd_input.value = jd_text
        if not jd_text:
            print("❌ Error: Please paste a Job Description or specify a valid file path before running generation.")
            return
            
        # Set environment variables for config and wiki directory
        os.environ["CONFIG_FILE"] = config_picker.value
        set_wiki_dir(wiki_dir_input.value.strip())
        
        print(f"🚀 Building graph with config: {config_picker.value}...")
        app = build_graph()
        
        inputs = {
            "job_description": jd_text,
            "job_description_raw": jd_text,
            "iteration_count": 0,
            "max_iterations": 3,
            "strategy_override": strategy_picker.value
        }
        
        logger.info("⏳ Running Agentic CV Generation Graph (Analysis -> Retrieval -> Drafting -> Audit)...")
        final_state = dict(inputs)
        try:
            for step_output in app.stream(inputs):
                for node_name, node_state in step_output.items():
                    if isinstance(node_state, dict):
                        final_state.update(node_state)
                        if node_name == "analyzer":
                            role = node_state.get("target_role", "N/A")
                            org = node_state.get("target_organization_slug", "N/A")
                            strat = node_state.get("target_region", "N/A")
                            logger.info("  🔍 [Node A: Analyzer] Completed -> Role: '%s', Org: '%s', Strategy: '%s'", role, org, strat)
                        elif node_name == "retriever":
                            exps = len(node_state.get("retrieved_experiences", []))
                            skills = len(node_state.get("skills_entries", []))
                            logger.info("  📦 [Node B: Retriever] Completed -> Selected %d experiences, %d skills", exps, skills)
                        elif node_name == "drafter":
                            draft_len = len(node_state.get("draft_cv", ""))
                            logger.info("  ✍️ [Node C: Drafter] Completed -> Draft CV generated (%d chars)", draft_len)
                        elif node_name == "refiner":
                            fb = node_state.get("refiner_feedback", "")
                            msg = "Page budget exceeded -> routing to compressor" if fb else "Page budget OK"
                            logger.info("  📐 [Refiner Guard] %s", msg)
                        elif node_name == "compressor":
                            pass_num = node_state.get("compression_count", 1)
                            logger.info("  🗜️ [Compressor] Applied compression pass %d/2", pass_num)
                        elif node_name == "auditor":
                            score = node_state.get("ats_scorecard", {}).get("total_score", "N/A")
                            fb = node_state.get("audit_feedback", "")
                            it = node_state.get("iteration_count", 1)
                            verdict = "PASS" if "PASS" in fb else "REWRITE REQUESTED"
                            logger.info("  🧐 [Node D: Auditor] Iteration %d/3 -> ATS Score: %s/100 | Verdict: %s", it, score, verdict)
                            if "PASS" not in fb and it < 3:
                                logger.warning("  🔄 Looping back to Drafter with auditor feedback...")
        except Exception as e:
            logger.error("❌ Pipeline failed with error: %s", e)
            return
            
        draft_cv = final_state.get("draft_cv", "")
        print(f"✅ CV Generation Complete! Audit iterations required: {final_state.get('iteration_count')}")
        
        # Save outputs
        out_filename = output_name_input.value.strip() or "Notebook_Output_CV.md"
        out_dir = validate_path(get_output_dir())
        out_dir.mkdir(exist_ok=True)
        out_path = out_dir / out_filename
        
        # Automatically save raw Job Description text to job-descriptions/ directory
        jd_dir = validate_path(Path("job-descriptions"))
        jd_dir.mkdir(exist_ok=True)
        jd_file_path = jd_dir / f"{out_path.stem}_jd.txt"
        jd_file_path.write_text(jd_text, encoding="utf-8")
        print(f"📥 Job Description text saved to: {jd_file_path}")
        
        out_path.write_text(draft_cv, encoding="utf-8")
        print(f"✨ Markdown CV saved to: {out_path}")
        
        if pdf_checkbox.value:
            pdf_path = out_path.with_suffix(".pdf")
            success = generate_pdf(draft_cv, str(pdf_path), "templates/base.css")
            if success:
                print(f"📄 PDF saved to: {pdf_path}")
            else:
                print("❌ PDF generation failed.")
                
        if docx_checkbox.value:
            docx_path = out_path.with_suffix(".docx")
            success = generate_docx(draft_cv, str(docx_path))
            if success:
                print(f"📝 DOCX saved to: {docx_path}")
            else:
                print("❌ Word (docx) generation failed.")
                
        print("\n" + "="*60 + "\n📌 GENERATED CV PREVIEW:\n" + "="*60)
        display(Markdown(draft_cv))

generate_button.on_click(on_generate_clicked)

# Display Unified UI Layout
ui_layout = widgets.VBox([
    widgets.HTML("<h3>1. Pipeline Parameters</h3>"),
    config_picker,
    strategy_picker,
    wiki_dir_input,
    widgets.HBox([pdf_checkbox, docx_checkbox]),
    widgets.HTML("<h3>2. Target Job Description & Output</h3>"),
    widgets.HBox([jd_file_input, load_jd_file_btn]),
    jd_input,
    output_name_input,
    widgets.HTML("<br>"),
    generate_button,
    widgets.HTML("<hr>"),
    out_area
])

display(ui_layout)

[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.



-----

WeasyPrint could not import some external libraries. Please carefully follow the installation steps before reporting an issue:
https://doc.courtbouillon.org/weasyprint/stable/first_steps.html#installation
https://doc.courtbouillon.org/weasyprint/stable/first_steps.html#troubleshooting 

-----

